# Framing and checksums

Serial links need two things: a way to find frame boundaries (SLIP, COBS, HDLC) and a way to detect corruption (CRC). IoTCom.Net.Framing has both, allocation-free and streaming.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.3.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.3.0-preview.1"

## The CRC catalogue
Every preset is verified against its catalogue check value for ASCII `123456789`.

In [ ]:
using IoTCom.Net.Framing;
foreach (var crc in CrcCatalog.All)
    Console.WriteLine($"{crc.Parameters.Name,-18} 0x{crc.Compute("123456789"u8):X}  ok={crc.SelfTest()}");

## Same bytes, three framings

In [ ]:
using System.Buffers;
byte[] payload = [0x7E, 0xC0, 0x00, 0x11, 0xDB];
foreach (IFrameEncoder codec in new IFrameEncoder[] { new Slip(), new Cobs(), new Hdlc() })
{
    var w = new ArrayBufferWriter<byte>();
    codec.Encode(payload, w);
    Console.WriteLine($"{codec.GetType().Name,-5} {IoTCom.Net.HexDump.ToHex(w.WrittenSpan)}");
}

## Streaming decode
Decoders work on `ReadOnlySequence<byte>` straight from a `PipeReader`, so frames split across reads are handled.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*

In [ ]:
using System.IO.Pipelines;
var pipe = new Pipe();
var cobs = new Cobs();
await pipe.Writer.WriteFrameAsync(cobs, new byte[] { 1, 0, 2 });
await pipe.Writer.WriteFrameAsync(cobs, new byte[] { 3, 4 });
await pipe.Writer.CompleteAsync();
await foreach (var frame in pipe.Reader.ReadFramesAsync(cobs)) Console.WriteLine(IoTCom.Net.HexDump.ToHex(frame));